# 05. Uncertainty Quantification & Conformal Prediction

**Course:** URP4201 Research Methodology

Platt scaling probability calibration and Inductive Conformal Prediction (LAC) confidence sets with finite-sample coverage guarantees.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "AquaSense AI" else Path(".").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import joblib
import numpy as np
import pandas as pd

from src.aquasense.uncertainty import assess_prediction_confidence

bundle = joblib.load("models/model_bundle.joblib")
clf = bundle["model"]
quantile = float(bundle["conformal_quantile"])
threshold = max(0.0, 1.0 - quantile)
classes = ["Good", "Moderate", "Polluted"]

print("Conformal Non-Conformity Quantile (alpha = 0.10):", round(quantile, 4))
print(f"Class Membership Inclusion Threshold (1 - q_hat): {threshold:.4f}")

Conformal Non-Conformity Quantile (alpha = 0.10): 0.023
Class Membership Inclusion Threshold (1 - q_hat): 0.9770


In [2]:
# Conformal Marginal Coverage Evaluation on Holdout Split
with open("results/metrics.json", encoding="utf-8") as f:
    conf_eval = json.load(f)["conformal_uncertainty"]

pd.DataFrame([{
    "Target Marginal Coverage (%)": conf_eval["target_coverage"] * 100,
    "Empirical Holdout Coverage (%)": round(conf_eval["empirical_coverage"] * 100, 2),
    "Average Prediction Set Size": round(conf_eval["average_set_size"], 2),
    "Quantile Threshold": round(conf_eval["quantile_threshold"], 4)
}])

,Target Marginal Coverage (%),Empirical Holdout Coverage (%),Average Prediction Set Size,Quantile Threshold
0,90.0,99.5,1.0,0.023


In [3]:
# Evaluation across Distinct Confidence Profiles: Demonstrating Conformal Set Behavior
profiles = {
    "High Confidence Good": [7.3, 2.0, 21.0, 7.5, 1.2, 0.002, 0.0001, 0.002],
    "High Confidence Polluted": [4.5, 42.0, 29.0, 1.8, 22.0, 0.06, 0.005, 0.04],
    "Borderline / Ambiguous": [6.3, 5.8, 24.0, 5.9, 2.8, 0.010, 0.0009, 0.009],
}

rows = []
for label, vals in profiles.items():
    x = np.array(vals, dtype=float).reshape(1, -1)
    probs = clf.predict_proba(x)[0]
    pred = int(np.argmax(probs))
    c_set = [classes[i] for i, p in enumerate(probs) if p >= threshold]
    needs_lab, note = assess_prediction_confidence(probs, [i for i, p in enumerate(probs) if p >= threshold])
    rows.append({
        "Sample Profile": label,
        "Screening Tier": classes[pred],
        "Prob Good": round(probs[0], 4),
        "Prob Moderate": round(probs[1], 4),
        "Prob Polluted": round(probs[2], 4),
        "Conformal Set": str(c_set),
        "Needs Lab Test": needs_lab,
        "Confidence Note": note
    })

pd.DataFrame(rows)

,Sample Profile,Screening Tier,Prob Good,Prob Moderate,Prob Polluted,Conformal Set,Needs Lab Test,Confidence Note
0,High Confidence Good,Good,0.9833,0.0090,0.0077,['Good'],False,Sufficient screening confidence.
1,High Confidence Polluted,Polluted,0.0100,0.0091,0.9809,['Polluted'],False,Sufficient screening confidence.
2,Borderline / Ambiguous,Moderate,0.0099,0.9825,0.0077,['Moderate'],False,Sufficient screening confidence.
